In [ ]:
import sys
import pandas as pd
import geopandas as gpd
import os
from datetime import datetime

sys.path.append('..')  # Add parent directory to path

# Import from local utility modules
from lvt_utils import ensure_geodataframe
from cloud_utils import get_mapserver_data_with_geometry_pa

# Set pandas to display all columns
pd.set_option('display.max_columns', None)

# Define Lackawanna County Parcels MapServer endpoint components
dataset_name = "GISViewer/Parcels"
base_url = "https://gis.lackawannacounty.org/arcgis/rest/services"
layer_id = 0

# Set scrape variable as needed
data_scrape = 1  # set to 0 or 1 as required

save_dir = os.path.join("data", "scranton")
os.makedirs(save_dir, exist_ok=True)

if data_scrape == 1:
    # Download data with geometry (paginate=True to pull all records)
    scranton_gdf = get_mapserver_data_with_geometry_pa(dataset_name, base_url, layer_id, paginate=True)
    # Make sure it's a GeoDataFrame
    scranton_gdf = ensure_geodataframe(scranton_gdf)
    # Compose filename with date
    today_str = datetime.now().strftime("%Y%m%d")
    fname = f"scranton_{today_str}.gpq"
    fpath = os.path.join(save_dir, fname)
    scranton_gdf.to_parquet(fpath)
    display(scranton_gdf.head())
else:
    # Find most recent geoparquet in data/scranton
    try:
        files = [f for f in os.listdir(save_dir) if f.lower().endswith(".gpq")]
        if not files:
            raise FileNotFoundError("No scranton geoparquet scrapes found in data/scranton/")

        # Get the latest by date in filename
        files_with_dates = []
        for fname in files:
            try:
                date_str = fname.split("_")[1].split(".")[0]
                dt = datetime.strptime(date_str, "%Y%m%d")
                files_with_dates.append((dt, fname))
            except Exception:
                continue
        if not files_with_dates:
            raise FileNotFoundError("No valid scranton geoparquet scrapes found in data/scranton/")
        latest_fname = max(files_with_dates, key=lambda x: x[0])[1]
        fpath = os.path.join(save_dir, latest_fname)
        scranton_gdf = gpd.read_parquet(fpath)
        display(scranton_gdf.head())
    except Exception as e:
        raise RuntimeError(f"Failed to find any previous scraped data: {e}")


In [ ]:
# TODO: Export standardized CSV — scranton
# This cell needs to be completed manually before running.
#
# Scranton notebook only fetches data (1 code cell) — no modeling implemented yet. Complete the modeling first, then add the export cell.
#
# Template:
# import sys
# sys.path.insert(0, '../..')
# from lvt_utils import save_standard_export
#
# save_standard_export(
#     df=<final_df_variable>,
#     city='scranton',
#     output_path='../../analysis/data/scranton.csv',
#     model_type='split_rate:4.0',  # update as needed
#     land_millage=land_millage,
#     improvement_millage=improvement_millage,
# )
